In [1]:
# Import the drive module from google.colab to enable Google Drive access
from google.colab import drive

# Import pandas for downloading, parsing, and inspecting tabular datasets
import pandas as pd

# Mount Google Drive to the default '/content/drive' directory
drive.mount('/content/drive')

# Store the URL for the NOAA CPC ONI (Oceanic Niño Index) ASCII file
oni_url = 'https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt'

# Store the URL for the NOAA CPC RONI (Relative Ocean Niño Index) ASCII file
roni_url = 'https://www.cpc.ncep.noaa.gov/data/indices/RONI.ascii.txt'

# Load the whitespace-delimited ONI data into a pandas DataFrame
oni_df = pd.read_csv(oni_url, sep=r'\s+')

# Load the whitespace-delimited RONI data into a pandas DataFrame
roni_df = pd.read_csv(roni_url, sep=r'\s+')

# Print header section for ONI column names
print('=== ONI COLUMN NAMES ===')

# Print the list of column names in the ONI DataFrame
print(oni_df.columns.tolist())

# Print header section for the first 5 rows of the ONI DataFrame
print('\n=== ONI FIRST 5 ROWS ===')

# Display the top 5 rows of the ONI DataFrame
print(oni_df.head(5))

# Print header section for the last 5 rows of the ONI DataFrame
print('\n=== ONI LAST 5 ROWS ===')

# Display the bottom 5 rows of the ONI DataFrame
print(oni_df.tail(5))

# Print header section for RONI column names
print('\n=== RONI COLUMN NAMES ===')

# Print the list of column names in the RONI DataFrame
print(roni_df.columns.tolist())

# Print header section for the first 5 rows of the RONI DataFrame
print('\n=== RONI FIRST 5 ROWS ===')

# Display the top 5 rows of the RONI DataFrame
print(roni_df.head(5))

# Print header section for the last 5 rows of the RONI DataFrame
print('\n=== RONI LAST 5 ROWS ===')

# Display the bottom 5 rows of the RONI DataFrame
print(roni_df.tail(5))

Mounted at /content/drive
=== ONI COLUMN NAMES ===
['SEAS', 'YR', 'TOTAL', 'ANOM']

=== ONI FIRST 5 ROWS ===
  SEAS    YR  TOTAL  ANOM
0  DJF  1950  25.01 -1.32
1  JFM  1950  25.36 -1.20
2  FMA  1950  25.88 -1.12
3  MAM  1950  26.24 -1.08
4  AMJ  1950  26.35 -1.10

=== ONI LAST 5 ROWS ===
    SEAS    YR  TOTAL  ANOM
915  MAM  2026  28.09  0.46
916  AMJ  2026  28.74  0.95
917  MJJ  2026  29.02  1.39
918  JJA  2026  29.09  1.80
919  JAS  2026  29.12  2.16

=== RONI COLUMN NAMES ===
['SEAS', 'YR', 'ANOM']

=== RONI FIRST 5 ROWS ===
  SEAS    YR  ANOM
0  DJF  1950 -1.19
1  JFM  1950 -1.08
2  FMA  1950 -0.96
3  MAM  1950 -1.00
4  AMJ  1950 -0.99

=== RONI LAST 5 ROWS ===
    SEAS    YR  ANOM
915  MAM  2026 -0.04
916  AMJ  2026  0.49
917  MJJ  2026  0.97
918  JJA  2026  1.36
919  JAS  2026  1.69


In [2]:
# Import os module to manage file paths and directory creation
import os

# Import pandas library for data manipulation and analysis
import pandas as pd

# Filter oni_df for JJA season and years 1981 through 2026
oni_jja = oni_df[
    (oni_df['SEAS'] == 'JJA') & (oni_df['YR'] >= 1981) & (oni_df['YR'] <= 2026)
][['YR', 'ANOM']].copy()

# Rename ANOM column to ONI_JJA for clarity
oni_jja = oni_jja.rename(columns={'ANOM': 'ONI_JJA'})

# Filter roni_df for JJA season and years 1981 through 2026
roni_jja = roni_df[
    (roni_df['SEAS'] == 'JJA') & (roni_df['YR'] >= 1981) & (roni_df['YR'] <= 2026)
][['YR', 'ANOM']].copy()

# Rename ANOM column to RONI_JJA for clarity
roni_jja = roni_jja.rename(columns={'ANOM': 'RONI_JJA'})

# Merge ONI and RONI JJA DataFrames on the year column (YR)
enso_jja = pd.merge(oni_jja, roni_jja, on='YR')


# Define helper function to classify ENSO phase from anomaly value
def classify_enso(anom):
  # Return 'El Niño' if anomaly is greater than or equal to 0.5
  if anom >= 0.5:
    # Return El Niño label
    return 'El Niño'
  # Return 'La Niña' if anomaly is less than or equal to -0.5
  elif anom <= -0.5:
    # Return La Niña label
    return 'La Niña'
  # Return 'Neutral' for values strictly between -0.5 and 0.5
  else:
    # Return Neutral label
    return 'Neutral'


# Create ONI_Class column by applying classification function to ONI_JJA
enso_jja['ONI_Class'] = enso_jja['ONI_JJA'].apply(classify_enso)

# Create RONI_Class column by applying classification function to RONI_JJA
enso_jja['RONI_Class'] = enso_jja['RONI_JJA'].apply(classify_enso)

# Filter DataFrame for years where ONI and RONI classifications disagree
disagreements = enso_jja[enso_jja['ONI_Class'] != enso_jja['RONI_Class']]

# Print section header for disagreement years
print('=== YEARS WHERE CLASSIFICATIONS DISAGREE ===')

# Output table of disagreement years without index column
print(disagreements.to_string(index=False))

# Filter DataFrame for selected benchmark years (1997, 2015, 2023, 2026)
selected_years = enso_jja[enso_jja['YR'].isin([1997, 2015, 2023, 2026])]

# Print section header for selected benchmark years
print('\n=== ROWS FOR 1997, 2015, 2023, AND 2026 ===')

# Output table of selected years without index column
print(selected_years.to_string(index=False))

# Define output directory path in Google Drive
output_dir = '/content/drive/MyDrive/ENSO_project'

# Ensure output directory exists before writing file
os.makedirs(output_dir, exist_ok=True)

# Define full CSV file output path
output_file = os.path.join(output_dir, 'enso_jja_classes.csv')

# Export complete table to CSV file without DataFrame index
enso_jja.to_csv(output_file, index=False)

# Print confirmation message showing file save location
print(f'\nSuccessfully saved table to {output_file}')

=== YEARS WHERE CLASSIFICATIONS DISAGREE ===
  YR  ONI_JJA  RONI_JJA ONI_Class RONI_Class
1992     0.35      0.52   Neutral    El Niño
1994     0.30      0.53   Neutral    El Niño
2004     0.39      0.54   Neutral    El Niño
2007    -0.49     -0.50   Neutral    La Niña
2009     0.50      0.35   El Niño    Neutral
2016    -0.19     -0.70   Neutral    La Niña
2020    -0.29     -0.76   Neutral    La Niña
2021    -0.35     -0.60   Neutral    La Niña

=== ROWS FOR 1997, 2015, 2023, AND 2026 ===
  YR  ONI_JJA  RONI_JJA ONI_Class RONI_Class
1997     1.48      1.50   El Niño    El Niño
2015     1.44      1.27   El Niño    El Niño
2023     1.00      0.54   El Niño    El Niño
2026     1.80      1.36   El Niño    El Niño

Successfully saved table to /content/drive/MyDrive/ENSO_project/enso_jja_classes.csv
